In [2]:
import cv2
import os


# paths
INPUT_FOLDER = r"N:\Final Preparation\FULL_DATASET\new\testing\positive\standardized_vedio"
OUTPUT_DIR   = r"N:\Final Preparation\FULL_DATASET\new\testing\positive\standardized_vedio\frame"

SAVE_EVERY_N = 1
IMAGE_FORMAT = "jpg"
VIDEO_EXTENSIONS = ('.mp4', '.avi', '.mov', '.mkv', '.wmv')

SORTED_PREFIX = "normal"   # normal_001, normal_002 ...
START_NUMBER  = 300


class BatchFrameExtractor:

    def __init__(self, input_folder, output_dir, save_every_n=1,
                 image_format="jpg", sorted_prefix="normal", start_number=1):
        self.input_folder  = input_folder
        self.output_dir    = output_dir
        self.save_every_n  = save_every_n
        self.image_format  = image_format
        self.sorted_prefix = sorted_prefix
        self.start_number  = start_number

        self.frames_dir = os.path.join(output_dir, "frames")
        self.sorted_dir = os.path.join(output_dir, "sorted")

    def _get_video_files(self):
        return sorted([
            f for f in os.listdir(self.input_folder)
            if f.lower().endswith(VIDEO_EXTENSIONS)
        ])

    def _extract(self, video_path, output_folder, video_name):
        os.makedirs(output_folder, exist_ok=True)

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise IOError(f"Cannot open video: {video_name}")

        saved = frame_idx = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % self.save_every_n == 0:
                filename = f"{video_name}_frame_{frame_idx:06d}.{self.image_format}"
                cv2.imwrite(os.path.join(output_folder, filename), frame)
                saved += 1
            frame_idx += 1

        cap.release()
        return saved

    def _save_mapping(self, folder_map):
        map_file = os.path.join(self.output_dir, "folder_mapping.txt")
        with open(map_file, "w") as f:
            f.write(f"{'Video Name':<35} {'Sorted Folder':<15} {'Frames':>7}\n")
            f.write(f"{'-'*57}\n")
            for video_name, sorted_name, count in folder_map:
                f.write(f"{video_name:<35} {sorted_name:<15} {count:>7}\n")
        print(f"Mapping saved -> {map_file}")

    def run(self):
        video_files = self._get_video_files()

        if not video_files:
            print(f"No videos found in: {self.input_folder}")
            return

        os.makedirs(self.frames_dir, exist_ok=True)
        os.makedirs(self.sorted_dir, exist_ok=True)

        print(f"Found {len(video_files)} video(s), starting...\n")

        total_saved = 0
        failed = []
        folder_map = []

        for i, filename in enumerate(video_files, start=1):
            video_name  = os.path.splitext(filename)[0]
            video_path  = os.path.join(self.input_folder, filename)
            frames_out  = os.path.join(self.frames_dir, video_name)
            sorted_name = f"{self.sorted_prefix}_{self.start_number + i - 1:03d}"
            sorted_out  = os.path.join(self.sorted_dir, sorted_name)
            os.makedirs(sorted_out, exist_ok=True)

            print(f"[{i}/{len(video_files)}] {video_name}", end=" ... ", flush=True)

            try:
                saved = self._extract(video_path, frames_out, video_name)
                total_saved += saved
                folder_map.append((video_name, sorted_name, saved))
                print(f"{saved} frames saved  ->  sorted/{sorted_name}/")
            except Exception as e:
                failed.append(video_name)
                folder_map.append((video_name, sorted_name, 0))
                print(f"FAILED ({e})")

        print(f"\nDone. {len(video_files) - len(failed)}/{len(video_files)} videos | {total_saved} total frames")

        if failed:
            print(f"Failed: {', '.join(failed)}")

        self._save_mapping(folder_map)


extractor = BatchFrameExtractor(
    input_folder  = INPUT_FOLDER,
    output_dir    = OUTPUT_DIR,
    save_every_n  = SAVE_EVERY_N,
    image_format  = IMAGE_FORMAT,
    sorted_prefix = SORTED_PREFIX,
    start_number  = START_NUMBER,
)
extractor.run()

Found 9 video(s), starting...

[1/9] theft_071 ... 128 frames saved  ->  sorted/normal_300/
[2/9] theft_074 ... 90 frames saved  ->  sorted/normal_301/
[3/9] theft_075 ... 51 frames saved  ->  sorted/normal_302/
[4/9] theft_076 ... 142 frames saved  ->  sorted/normal_303/
[5/9] theft_077 ... 105 frames saved  ->  sorted/normal_304/
[6/9] theft_078 ... 141 frames saved  ->  sorted/normal_305/
[7/9] theft_079 ... 119 frames saved  ->  sorted/normal_306/
[8/9] theft_080 ... 125 frames saved  ->  sorted/normal_307/
[9/9] theft_082 ... 111 frames saved  ->  sorted/normal_308/

Done. 9/9 videos | 1012 total frames
Mapping saved -> N:\Final Preparation\FULL_DATASET\new\testing\positive\standardized_vedio\frame\folder_mapping.txt
